## CATBOOST

## catboost funzionante 

In [1]:
import pandas as pd

df = pd.read_csv("vehicles_dataset.csv")

In [2]:
df = df.query("500 <= price <= 100000")

In [3]:
df = df.query("500 <= odometer <= 400000")

In [4]:
df.info()

<class 'pandas.DataFrame'>
Index: 373866 entries, 27 to 426879
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   price         373866 non-null  int64  
 1   year          372921 non-null  float64
 2   manufacturer  359990 non-null  str    
 3   model         369798 non-null  str    
 4   condition     232808 non-null  str    
 5   cylinders     222263 non-null  str    
 6   fuel          371668 non-null  str    
 7   odometer      373866 non-null  float64
 8   title_status  367561 non-null  str    
 9   transmission  372415 non-null  str    
 10  drive         261313 non-null  str    
 11  size          105938 non-null  str    
 12  type          294826 non-null  str    
 13  paint_color   267148 non-null  str    
 14  state         373866 non-null  str    
dtypes: float64(2), int64(1), str(12)
memory usage: 45.6 MB


In [5]:
categorical_cols = [
    "manufacturer", "model", "condition", "cylinders", "fuel",
    "title_status", "transmission", "drive", "size", "type",
    "paint_color", "state"
]

In [6]:
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

In [7]:
df["year"] = df["year"].fillna(df["year"].median())
df["odometer"] = df["odometer"].fillna(df["odometer"].median())


In [8]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [9]:
for col in categorical_cols:
    X_train[col] = X_train[col].astype("string")
    X_test[col] = X_test[col].astype("string")


In [10]:
from catboost import CatBoostRegressor
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    cat_features=categorical_cols
)


0:	learn: 13875.2060380	test: 13807.5450476	best: 13807.5450476 (0)	total: 380ms	remaining: 1m 53s
200:	learn: 6467.2674548	test: 6346.5290192	best: 6346.5290192 (200)	total: 45.3s	remaining: 22.3s
299:	learn: 6253.4536101	test: 6139.3707785	best: 6139.3707785 (299)	total: 1m 9s	remaining: 0us

bestTest = 6139.370778
bestIteration = 299



CatBoostRegressor(depth=6, eval_metric='RMSE', iterations=300, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=200)

In [11]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R²:", r2)


MAE: 3746.777255298367
MSE: 37691873.55555759
RMSE: 6139.370778472138
R²: 0.8134381300905643
